# Task 1 – Visualizing the Mechanics

The probability that a user with latent ability $\theta$ correctly answers item $i$ is modelled using the Two-Parameter Logistic (2PL) Item Response Theory model,

$$
P(Y_i=1\mid\Theta=\theta)=p_i(\theta)
=
\frac{1}{1+\exp\left[-a_i(\theta-b_i)\right]}.
$$

where

- $\theta$ is the user's latent ability,
- $a_i>0$ is the discrimination parameter,
- $b_i$ is the difficulty parameter.

The discrimination parameter $a_i$ determines how sensitive the probability of a correct response is to changes in the user's ability.

- A larger value of $a_i$ produces a steeper logistic curve, making the item more effective at distinguishing between users with different ability levels.
- A smaller value of $a_i$ produces a flatter curve, indicating that the item provides less information about the user's ability.

The difficulty parameter $b_i$ controls the horizontal location of the curve.

- Increasing $b_i$ shifts the curve to the right, meaning that a higher ability level is required to achieve the same probability of answering correctly.
- Decreasing $b_i$ shifts the curve to the left, making the item easier.

In this task, two different discrimination values are considered. One discrimination value is paired with three different difficulty values to illustrate how changing $b_i$ shifts the Item Response Function horizontally while changing $a_i$ alters its steepness.

## Interpretation

From the plotted curves, it can be observed that:

- Increasing the discrimination parameter $a_i$ makes the transition from low probability to high probability much steeper.
- Increasing the difficulty parameter $b_i$ shifts the entire curve towards higher values of $\theta$, indicating that more ability is required to answer the item correctly.
- Decreasing the difficulty parameter shifts the curve towards lower values of $\theta$, making the item easier.

Thus, the discrimination parameter controls the **steepness** of the Item Response Curve, whereas the difficulty parameter controls its **horizontal position**.

In [1]:
import numpy as np
import plotly.graph_objects as go

# -------------------------------------------------------
# Two-Parameter Logistic (2PL) Item Response Function
# -------------------------------------------------------
def p_i(theta, a, b):
    return 1 / (1 + np.exp(-a * (theta - b)))

# Generate latent ability values
theta = np.linspace(-6, 6, 500)

# Curves required by the assignment
curves = [
    {"a": 0.5, "b": 0},
    {"a": 1.5, "b": -2},
    {"a": 1.5, "b": 0},
    {"a": 1.5, "b": 2},
]

# Create figure
fig = go.Figure()

# Plot each Item Response Curve
for curve in curves:
    a = curve["a"]
    b = curve["b"]

    fig.add_trace(
        go.Scatter(
            x=theta,
            y=p_i(theta, a, b),
            mode="lines",
            name=f"a = {a}, b = {b}",
            line=dict(width=3)
        )
    )

# Layout
fig.update_layout(
    title="Two-Parameter Logistic (2PL) Item Response Curves",
    xaxis_title="Latent Ability (θ)",
    yaxis_title="Probability of Correct Response",
    xaxis=dict(range=[-6, 6]),
    yaxis=dict(range=[0, 1.05]),
    template="plotly_white",
    hovermode="x unified",
    width=900,
    height=550
)

fig.show()

# Task 2 – Sequential Likelihood Contribution

Let the response to the $k$-th item be denoted by

$$
Y_k=
\begin{cases}
1, & \text{if the user answers the item correctly},\\
0, & \text{if the user answers the item incorrectly}.
\end{cases}
$$

According to the Two-Parameter Logistic (2PL) model, the probability that the user answers item $k$ correctly is

$$
p_k(\theta)
=
\frac{1}{1+\exp\left[-a_k(\theta-b_k)\right]},
$$

where

- $\theta$ is the user's latent ability,
- $a_k$ is the discrimination parameter,
- $b_k$ is the difficulty parameter.

The likelihood contribution of a single observed response is

$$
L(y_k|\theta)
=
p_k(\theta)^{y_k}
\left(1-p_k(\theta)\right)^{1-y_k}.
$$

If the user answers correctly ($y_k=1$), then

$$
L(y_k|\theta)=p_k(\theta).
$$

If the user answers incorrectly ($y_k=0$), then

$$
L(y_k|\theta)=1-p_k(\theta).
$$

Assuming that all item responses are conditionally independent given the latent ability $\theta$, the joint likelihood of the observed response history

$$
\mathbf{y}^{(k)}
=
(y_1,y_2,\ldots,y_k)
$$

is

$$
L(\mathbf{y}^{(k)}|\theta)
=
\prod_{i=1}^{k}
p_i(\theta)^{y_i}
\left(1-p_i(\theta)\right)^{1-y_i}.
$$

The joint likelihood represents the probability of observing the complete sequence of responses for a given value of the latent ability parameter. This likelihood combines all information collected up to step $k$ and is used together with the prior distribution to obtain the posterior distribution through Bayes' theorem.

# Task 3 – Mathematical Formulation of the Running Update

Bayesian inference updates the estimate of the user's latent ability whenever a new response is observed. At each step, the posterior distribution is obtained by combining the likelihood of the latest response with the posterior distribution from the previous step.

Let the response history after $k$ items be

$$
\mathbf{y}^{(k)}
=
(y_1,y_2,\ldots,y_k).
$$

Using Bayes' theorem, the posterior distribution after observing the $k$-th response is

$$
f_{\Theta|Y^{(k)}}(\theta|\mathbf{y}^{(k)})
=
\frac{
L(y_k|\theta)\,
f_{\Theta|Y^{(k-1)}}(\theta|\mathbf{y}^{(k-1)})
}{
\displaystyle
\int_{-\infty}^{\infty}
L(y_k|s)\,
f_{\Theta|Y^{(k-1)}}(s|\mathbf{y}^{(k-1)})\,ds
}.
$$

The denominator is a normalising constant that ensures the posterior density integrates to one.

Ignoring the normalising constant, the recursive Bayesian update can be written as

$$
f_{\Theta|Y^{(k)}}(\theta|\mathbf{y}^{(k)})
\propto
L(y_k|\theta)\,
f_{\Theta|Y^{(k-1)}}(\theta|\mathbf{y}^{(k-1)}).
$$

This expression shows that the posterior distribution at step $k$ is proportional to

- the likelihood of the newly observed response, and
- the posterior distribution from the previous step.

Before observing any responses, the user's latent ability is assumed to follow a standard normal distribution,

$$
\Theta \sim N(0,1).
$$

Therefore, the initial prior density is

$$
f_{\Theta}^{(0)}(\theta)
=
\frac{1}{\sqrt{2\pi}}
\exp\left(-\frac{\theta^{2}}{2}\right).
$$

After the first response, this prior is updated using the likelihood of the observed response to obtain the first posterior distribution. The resulting posterior then becomes the prior for the next response. This sequential updating process continues throughout the assessment, allowing the estimate of the user's latent ability to improve as more responses are collected.

## Interpretation

The recursive Bayesian update enables the learning platform to refine its estimate of the user's ability after every answered question. Instead of analysing all responses from the beginning each time, the previous posterior distribution is retained and combined only with the newest response. This makes the updating process computationally efficient while ensuring that all past information is incorporated into the current estimate.

# Task 4 – Dynamic Shifting

The posterior distribution changes dynamically after each observed response because every new answer provides additional information about the user's latent ability. The direction and magnitude of this change depend on both the user's response and the characteristics of the current item.

Suppose the user answers a highly difficult item correctly. Such an item has a relatively large difficulty parameter $b_k$, meaning that only users with higher ability levels are likely to answer it correctly. The likelihood function for a correct response is

$$
L(y_k=1|\theta)=p_k(\theta),
$$

where

$$
p_k(\theta)
=
\frac{1}{1+\exp\left[-a_k(\theta-b_k)\right]}.
$$

The posterior distribution is updated according to

$$
f_{\Theta|Y^{(k)}}(\theta|\mathbf{y}^{(k)})
\propto
p_k(\theta)\,
f_{\Theta|Y^{(k-1)}}(\theta|\mathbf{y}^{(k-1)}).
$$

Since the probability of answering a difficult item correctly is higher for users with larger values of $\theta$, the likelihood assigns greater weight to higher ability values. Consequently, the posterior distribution shifts towards the right, indicating an increase in the estimated latent ability.

On the other hand, if the user answers the same difficult item incorrectly, the likelihood becomes

$$
L(y_k=0|\theta)
=
1-p_k(\theta).
$$

The posterior update becomes

$$
f_{\Theta|Y^{(k)}}(\theta|\mathbf{y}^{(k)})
\propto
\left(1-p_k(\theta)\right)
f_{\Theta|Y^{(k-1)}}(\theta|\mathbf{y}^{(k-1)}).
$$

In this case, smaller values of $\theta$ receive relatively greater weight, causing the posterior distribution to shift towards the left. This indicates that the estimated ability of the user decreases.

The size of the shift also depends on the discrimination parameter $a_k$. Items with larger discrimination values produce stronger evidence about the user's ability and therefore result in a larger movement of the posterior distribution. Items with smaller discrimination values produce weaker evidence, so the posterior changes more gradually.

## Interpretation

A correct response to a highly difficult item provides strong evidence that the user has a high ability level, causing the peak of the posterior distribution to move towards larger values of $\theta$. Conversely, an incorrect response to the same item shifts the posterior towards lower ability values. Therefore, Bayesian updating continuously adjusts the estimated ability by incorporating both the observed response and the characteristics of each item.

# Task 5 – Tracking Certainty and Sharpness

The discrimination parameter $a_k$ determines how effectively an item distinguishes between users with different ability levels. It controls the steepness of the Item Response Function (IRF) and therefore influences the amount of information gained from each observed response.

The probability of answering item $k$ correctly is given by the 2PL model,

$$
p_k(\theta)
=
\frac{1}{1+\exp\left[-a_k(\theta-b_k)\right]}.
$$

The likelihood of the observed response is

$$
L(y_k|\theta)
=
p_k(\theta)^{y_k}
\left(1-p_k(\theta)\right)^{1-y_k}.
$$

Using Bayes' theorem, the posterior distribution is updated as

$$
f_{\Theta|Y^{(k)}}(\theta|\mathbf{y}^{(k)})
\propto
L(y_k|\theta)\,
f_{\Theta|Y^{(k-1)}}(\theta|\mathbf{y}^{(k-1)}).
$$

## Effect of a Large Discrimination Parameter

When the discrimination parameter $a_k$ is large, the Item Response Function becomes much steeper around the difficulty parameter $b_k$. A small change in the user's ability results in a significant change in the probability of answering the item correctly.

Consequently,

- the likelihood function becomes highly concentrated around the most probable ability values,
- the observed response provides strong evidence about the user's ability,
- the posterior distribution becomes narrower,
- the posterior variance decreases,
- confidence in the estimated ability increases.

Therefore, highly discriminative items allow the Bayesian estimation process to learn the user's ability more rapidly.

## Effect of a Small Discrimination Parameter

When the discrimination parameter $a_k$ is small, the Item Response Function becomes flatter. The probability of a correct response changes gradually with the user's ability.

As a result,

- the likelihood function is less informative,
- each observed response contributes only a small amount of new information,
- the posterior distribution changes slowly,
- the posterior variance remains relatively large,
- uncertainty about the user's ability remains higher.

Thus, weakly discriminative items produce only minor updates to the posterior distribution.

## Interpretation

The discrimination parameter directly influences the certainty of the Bayesian estimate.

- A **large** value of $a_k$ produces a sharper and more concentrated posterior distribution, indicating greater confidence in the estimated ability.

- A **small** value of $a_k$ produces a broader posterior distribution, indicating greater uncertainty about the user's true ability.

As additional responses are observed, especially from highly discriminative items, the posterior distribution becomes progressively narrower. This reduction in posterior variance reflects the increasing certainty of the Bayesian estimation process and leads to a more accurate estimate of the user's latent ability.

# Task 6 – Numerical Implementation Using a Fixed Grid

The posterior distribution of the user's latent ability under the Two-Parameter Logistic (2PL) model does not have a closed-form analytical solution. Therefore, a numerical approximation is performed using a fixed grid of ability values.

Assume that the latent ability parameter is represented by a set of equally spaced grid points,

$$
\theta_1,\theta_2,\ldots,\theta_m,
$$

covering a suitable interval such as

$$
-5 \leq \theta \leq 5.
$$

The Bayesian updating procedure is carried out sequentially after each observed response.

## Step 1 – Construct the Ability Grid

A dense grid of equally spaced ability values is generated over the interval

$$
[-5,5].
$$

A finer grid provides a more accurate approximation of the posterior distribution.

## Step 2 – Initialise the Prior Distribution

Before observing any responses, the user's latent ability is assumed to follow a standard normal distribution,

$$
\Theta \sim N(0,1).
$$

The corresponding prior density is

$$
f^{(0)}(\theta)
=
\frac{1}{\sqrt{2\pi}}
\exp\left(-\frac{\theta^2}{2}\right).
$$

The prior probability is evaluated at every grid point.

## Step 3 – Compute the Response Probability

For the observed item, the probability of a correct response is computed using the 2PL model,

$$
p_k(\theta)
=
\frac{1}
{1+\exp\left[-a_k(\theta-b_k)\right]}.
$$

This probability is calculated for every value of $\theta$ on the grid.

## Step 4 – Evaluate the Likelihood

The likelihood corresponding to the observed response is

$$
L(y_k|\theta)
=
p_k(\theta)^{y_k}
\left(1-p_k(\theta)\right)^{1-y_k}.
$$

If

$$
y_k=1,
$$

then

$$
L(y_k|\theta)=p_k(\theta).
$$

If

$$
y_k=0,
$$

then

$$
L(y_k|\theta)=1-p_k(\theta).
$$

## Step 5 – Update the Posterior Distribution

The posterior is obtained by multiplying the previous posterior distribution by the likelihood,

$$
f_{\text{new}}(\theta)
=
f_{\text{old}}(\theta)
\times
L(y_k|\theta).
$$

This produces an unnormalised posterior distribution.

## Step 6 – Normalise the Posterior

To obtain a valid probability density function, the posterior is divided by its total area,

$$
f_{\text{new}}(\theta)
=
\frac{
f_{\text{new}}(\theta)
}{
\displaystyle
\int
f_{\text{new}}(\theta)\,d\theta
}.
$$

The integral is evaluated numerically using the trapezoidal rule.

After normalisation,

$$
\int
f_{\text{new}}(\theta)\,d\theta
=
1.
$$

## Step 7 – Repeat the Procedure

The normalised posterior obtained after the current response becomes the prior distribution for the next item.

The same sequence of calculations is repeated until every response has been processed.

## Summary

The numerical Bayesian updating algorithm consists of the following steps:

1. Generate a fixed grid of ability values.
2. Initialise the standard normal prior distribution.
3. Compute the response probability using the 2PL model.
4. Evaluate the likelihood of the observed response.
5. Multiply the previous posterior by the likelihood.
6. Normalise the updated posterior using numerical integration.
7. Repeat the process for all remaining responses.

This fixed-grid approach provides an efficient numerical approximation of the posterior distribution and allows the user's latent ability to be updated sequentially as new responses become available.

In [2]:
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go

# ==========================================================
# Configuration
# ==========================================================

np.random.seed(42)

theta_true = 0.75      # True user ability
n_items = 20           # Number of questions

# Ability grid
theta_grid = np.linspace(-5, 5, 1000)

# Prior distribution N(0,1)
posterior = stats.norm.pdf(theta_grid, 0, 1)
posterior /= np.trapezoid(posterior, theta_grid)

# Store estimates
posterior_means = []
map_estimates = []

# ==========================================================
# 2PL Item Response Function
# ==========================================================

def irf(theta, a, b):
    return 1 / (1 + np.exp(-a * (theta - b)))

# ==========================================================
# Bayesian Updating
# ==========================================================

for k in range(n_items):

    # Random item parameters
    a = np.random.uniform(0.5, 2.0)
    b = np.random.normal(0, 1)

    # True probability of correct response
    p_true = irf(theta_true, a, b)

    # Simulated response
    y = np.random.binomial(1, p_true)

    # Likelihood over the grid
    p_grid = irf(theta_grid, a, b)

    likelihood = np.where(y == 1, p_grid, 1 - p_grid)

    # Bayesian update
    posterior *= likelihood

    # Normalise
    posterior /= np.trapezoid(posterior, theta_grid)

    # Posterior Mean
    posterior_mean = np.trapezoid(theta_grid * posterior, theta_grid)

    # MAP Estimate
    map_estimate = theta_grid[np.argmax(posterior)]

    posterior_means.append(posterior_mean)
    map_estimates.append(map_estimate)

# ==========================================================
# Plot Convergence
# ==========================================================

steps = np.arange(1, n_items + 1)

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=steps,
        y=posterior_means,
        mode="lines+markers",
        name="Posterior Mean"
    )
)

fig.add_trace(
    go.Scatter(
        x=steps,
        y=map_estimates,
        mode="lines+markers",
        name="MAP Estimate"
    )
)

fig.add_trace(
    go.Scatter(
        x=[1, n_items],
        y=[theta_true, theta_true],
        mode="lines",
        name="True Ability",
        line=dict(dash="dash")
    )
)

fig.update_layout(
    title="Sequential Bayesian Estimation of User Ability",
    xaxis_title="Item Number",
    yaxis_title="Estimated Ability",
    template="plotly_white",
    width=900,
    height=550
)

fig.show()

# ==========================================================
# Final Results
# ==========================================================

print(f"True Ability        : {theta_true:.3f}")
print(f"Posterior Mean      : {posterior_means[-1]:.3f}")
print(f"MAP Estimate        : {map_estimates[-1]:.3f}")

True Ability        : 0.750
Posterior Mean      : 0.380
MAP Estimate        : 0.375


# Task 7 – Analysis of the Results

The simulation demonstrates how Bayesian estimation updates the user's latent ability after each observed response. Initially, the posterior distribution is broad because only the prior information is available. Consequently, both the Posterior Mean and the MAP estimate may differ considerably from the user's true ability.

As additional item responses are incorporated, the posterior distribution becomes progressively more concentrated. This reduction in spread indicates that uncertainty about the user's ability decreases as more evidence is collected.

The Posterior Mean provides the expected value of the posterior distribution, while the MAP estimate corresponds to the ability value with the highest posterior probability. Although these estimates may differ during the early stages of the assessment, they gradually converge as more responses are observed.

The convergence plot shows that both estimators move towards the true ability value over time. After processing a sufficient number of items, the estimates become stable and fluctuate only slightly around the true ability.

These results demonstrate that sequential Bayesian updating effectively combines prior knowledge with newly observed responses, producing increasingly accurate and reliable estimates of the user's latent ability.

# Q. Bayesian Tracking of Click-Through Rates (CTR) via Conjugate Beta-Binomial Updates


# Task 1 – Visualising the Beta Prior Distribution

The click-through rate (CTR) of an advertisement is represented by the unknown parameter $\theta$, where

$$
0 \leq \theta \leq 1.
$$

Before observing any user clicks, the prior belief about the CTR is modelled using a Beta distribution,

$$
\Theta \sim \mathrm{Beta}(\alpha,\beta).
$$

The probability density function of the Beta distribution is

$$
f(\theta)
=
\frac{1}{B(\alpha,\beta)}
\theta^{\alpha-1}
(1-\theta)^{\beta-1},
\qquad
0 \leq \theta \leq 1,
$$

where

- $\alpha > 0$ and $\beta > 0$ are the shape parameters.
- $B(\alpha,\beta)$ is the Beta function that ensures the total probability equals one.

Three different Beta distributions are considered to illustrate how the values of $\alpha$ and $\beta$ influence the prior belief.

### Case 1: Uniform Prior

When

$$
\alpha=1,\qquad \beta=1,
$$

the distribution is

$$
\mathrm{Beta}(1,1).
$$

This distribution is uniform over the interval $[0,1]$, indicating that every possible value of the CTR is equally likely before any observations are made.

### Case 2: Low CTR Prior

When

$$
\alpha=2,\qquad \beta=8,
$$

the distribution is

$$
\mathrm{Beta}(2,8).
$$

Since $\beta$ is much larger than $\alpha$, most of the probability mass is concentrated near zero. This represents a prior belief that the advertisement is expected to have a relatively low click-through rate.

### Case 3: High CTR Prior

When

$$
\alpha=8,\qquad \beta=2,
$$

the distribution is

$$
\mathrm{Beta}(8,2).
$$

Because $\alpha$ is much larger than $\beta$, the density is concentrated near one. This represents a prior belief that the advertisement is likely to achieve a high click-through rate.

## Interpretation

The parameters $\alpha$ and $\beta$ determine the shape of the Beta distribution.

- Increasing $\alpha$ shifts the density towards larger values of $\theta$, indicating a stronger belief in a higher click-through rate.
- Increasing $\beta$ shifts the density towards smaller values of $\theta$, indicating a stronger belief in a lower click-through rate.
- When $\alpha=\beta=1$, the distribution is uniform and expresses complete prior uncertainty.

As user click data become available, these parameters are updated through Bayesian inference, allowing the prior distribution to evolve into the posterior distribution.

# Task 1 – Visualising the Beta Prior Distribution

The click-through rate (CTR) of an advertisement is represented by the unknown parameter $\theta$, where

$$
0 \leq \theta \leq 1.
$$

Before observing any user clicks, the prior belief about the CTR is modelled using a Beta distribution,

$$
\Theta \sim \mathrm{Beta}(\alpha,\beta).
$$

The probability density function of the Beta distribution is

$$
f(\theta)
=
\frac{1}{B(\alpha,\beta)}
\theta^{\alpha-1}
(1-\theta)^{\beta-1},
\qquad
0 \leq \theta \leq 1,
$$

where

- $\alpha > 0$ and $\beta > 0$ are the shape parameters.
- $B(\alpha,\beta)$ is the Beta function that ensures the total probability equals one.

Three different Beta distributions are considered to illustrate how the values of $\alpha$ and $\beta$ influence the prior belief.

### Case 1: Uniform Prior

When

$$
\alpha=1,\qquad \beta=1,
$$

the distribution is

$$
\mathrm{Beta}(1,1).
$$

This distribution is uniform over the interval $[0,1]$, indicating that every possible value of the CTR is equally likely before any observations are made.

### Case 2: Low CTR Prior

When

$$
\alpha=2,\qquad \beta=8,
$$

the distribution is

$$
\mathrm{Beta}(2,8).
$$

Since $\beta$ is much larger than $\alpha$, most of the probability mass is concentrated near zero. This represents a prior belief that the advertisement is expected to have a relatively low click-through rate.

### Case 3: High CTR Prior

When

$$
\alpha=8,\qquad \beta=2,
$$

the distribution is

$$
\mathrm{Beta}(8,2).
$$

Because $\alpha$ is much larger than $\beta$, the density is concentrated near one. This represents a prior belief that the advertisement is likely to achieve a high click-through rate.

## Interpretation

The parameters $\alpha$ and $\beta$ determine the shape of the Beta distribution.

- Increasing $\alpha$ shifts the density towards larger values of $\theta$, indicating a stronger belief in a higher click-through rate.
- Increasing $\beta$ shifts the density towards smaller values of $\theta$, indicating a stronger belief in a lower click-through rate.
- When $\alpha=\beta=1$, the distribution is uniform and expresses complete prior uncertainty.

As user click data become available, these parameters are updated through Bayesian inference, allowing the prior distribution to evolve into the posterior distribution.

In [3]:
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go

# Generate theta values
theta = np.linspace(0, 1, 500)

# Beta distributions
distributions = [
    (1, 1, "Beta(1,1)"),
    (2, 8, "Beta(2,8)"),
    (8, 2, "Beta(8,2)")
]

fig = go.Figure()

for alpha, beta, label in distributions:
    fig.add_trace(
        go.Scatter(
            x=theta,
            y=stats.beta.pdf(theta, alpha, beta),
            mode="lines",
            name=label,
            line=dict(width=3)
        )
    )

fig.update_layout(
    title="Beta Prior Distributions",
    xaxis_title="CTR (θ)",
    yaxis_title="Probability Density",
    template="plotly_white",
    width=900,
    height=550
)

fig.show()

# Task 2 – Sequential Bernoulli Likelihood

Assume that each advertisement impression results in one of two possible outcomes:

- $y_k = 1$, if the user clicks the advertisement.
- $y_k = 0$, if the user does not click the advertisement.

Each observation therefore follows a Bernoulli distribution with an unknown click-through rate (CTR) parameter $\theta$.

The probability mass function of a single Bernoulli observation is

$$
P(Y_k=y_k \mid \theta)
=
\theta^{y_k}(1-\theta)^{1-y_k},
$$

where

$$
0 \leq \theta \leq 1.
$$

## Likelihood of a Single Observation

The likelihood function corresponding to the observed outcome $y_k$ is

$$
L(y_k \mid \theta)
=
\theta^{y_k}(1-\theta)^{1-y_k}.
$$

For a click,

$$
y_k=1,
$$

the likelihood becomes

$$
L(y_k \mid \theta)=\theta.
$$

For a non-click,

$$
y_k=0,
$$

the likelihood becomes

$$
L(y_k \mid \theta)=1-\theta.
$$

## Joint Likelihood of Multiple Observations

Suppose the advertisement has been displayed $k$ times, producing the response sequence

$$
\mathbf{y}^{(k)}
=
(y_1,y_2,\ldots,y_k).
$$

Assuming that each observation is independent, the joint likelihood is

$$
L(\mathbf{y}^{(k)} \mid \theta)
=
\prod_{i=1}^{k}
\theta^{y_i}
(1-\theta)^{1-y_i}.
$$

Using the properties of exponents,

$$
L(\mathbf{y}^{(k)} \mid \theta)
=
\theta^{\sum_{i=1}^{k}y_i}
(1-\theta)^{k-\sum_{i=1}^{k}y_i}.
$$

Let

$$
C_k=\sum_{i=1}^{k}y_i
$$

denote the total number of clicks observed after $k$ impressions.

The likelihood can then be written as

$$
L(\mathbf{y}^{(k)} \mid \theta)
=
\theta^{C_k}
(1-\theta)^{k-C_k}.
$$

## Interpretation

The likelihood depends only on two quantities:

- the total number of clicks, $C_k$, and
- the total number of non-clicks, $k-C_k$.

As more impressions are observed, these sufficient statistics summarise all the information required to update the posterior distribution. This compact representation makes Bayesian updating efficient and naturally leads to the Beta-Binomial conjugate update developed in the following tasks.

# Task 3 – Beta-Binomial Conjugacy Proof

The prior distribution for the unknown click-through rate $\theta$ is assumed to follow a Beta distribution,

$$
\Theta \sim \mathrm{Beta}(\alpha_{k-1},\beta_{k-1}),
$$

with probability density function

$$
f(\theta \mid Y^{(k-1)})
=
\frac{1}{B(\alpha_{k-1},\beta_{k-1})}
\theta^{\alpha_{k-1}-1}
(1-\theta)^{\beta_{k-1}-1},
$$

where

- $\alpha_{k-1}>0$,
- $\beta_{k-1}>0$, and
- $B(\alpha_{k-1},\beta_{k-1})$ is the Beta function.

After observing the next impression, the response is

$$
y_k \in \{0,1\}.
$$

The likelihood of this single Bernoulli observation is

$$
L(y_k \mid \theta)
=
\theta^{y_k}
(1-\theta)^{1-y_k}.
$$

## Applying Bayes' Theorem

The posterior distribution is obtained by multiplying the likelihood and the prior,

$$
f(\theta \mid Y^{(k)})
\propto
L(y_k \mid \theta)
\,
f(\theta \mid Y^{(k-1)}).
$$

Substituting the likelihood and prior gives

$$
f(\theta \mid Y^{(k)})
\propto
\theta^{y_k}
(1-\theta)^{1-y_k}
\,
\theta^{\alpha_{k-1}-1}
(1-\theta)^{\beta_{k-1}-1}.
$$

Combining the powers of the same terms,

$$
f(\theta \mid Y^{(k)})
\propto
\theta^{\alpha_{k-1}+y_k-1}
(1-\theta)^{\beta_{k-1}+1-y_k-1}.
$$

This expression has exactly the same functional form as the Beta distribution,

$$
f(\theta)
\propto
\theta^{\alpha-1}
(1-\theta)^{\beta-1}.
$$

Therefore, the posterior distribution is also a Beta distribution,

$$
\Theta \mid Y^{(k)}
\sim
\mathrm{Beta}(\alpha_k,\beta_k),
$$

where the updated parameters are

$$
\alpha_k
=
\alpha_{k-1}+y_k,
$$

and

$$
\beta_k
=
\beta_{k-1}+(1-y_k).
$$

Including the normalising constant, the posterior density is

$$
f(\theta \mid Y^{(k)})
=
\frac{1}{B(\alpha_k,\beta_k)}
\theta^{\alpha_k-1}
(1-\theta)^{\beta_k-1}.
$$

## Interpretation

This derivation demonstrates the **conjugacy** between the Beta prior and the Bernoulli likelihood.

- If the new observation is a click ($y_k=1$), the parameter $\alpha$ increases by one while $\beta$ remains unchanged.
- If the new observation is a non-click ($y_k=0$), the parameter $\beta$ increases by one while $\alpha$ remains unchanged.

Since the posterior distribution remains within the Beta family after every Bayesian update, only the two parameters $\alpha$ and $\beta$ need to be updated sequentially. This property makes Bayesian tracking of the click-through rate computationally simple and highly efficient.

# Task 4 – Closed-Form Arithmetic Update Parameters

From the Beta-Bernoulli conjugacy result obtained in Task 3, the posterior distribution after observing the $k$-th impression is

$$
\Theta \mid Y^{(k)}
\sim
\mathrm{Beta}(\alpha_k,\beta_k),
$$

where the recursive update equations are

$$
\alpha_k
=
\alpha_{k-1}+y_k,
$$

and

$$
\beta_k
=
\beta_{k-1}+(1-y_k).
$$

These equations show that the posterior parameters can be updated using only the latest observation.

- If the user clicks the advertisement ($y_k=1$), then

$$
\alpha_k=\alpha_{k-1}+1,
$$

and

$$
\beta_k=\beta_{k-1}.
$$

- If the user does not click the advertisement ($y_k=0$), then

$$
\alpha_k=\alpha_{k-1},
$$

and

$$
\beta_k=\beta_{k-1}+1.
$$

Thus, each click increases the parameter $\alpha$, while each non-click increases the parameter $\beta$.

## Closed-Form Expressions

After observing $k$ impressions, let

$$
C_k=\sum_{i=1}^{k}y_i
$$

represent the total number of clicks.

Since the number of non-clicks is

$$
k-C_k,
$$

the recursive updates can be written in closed form as

$$
\alpha_k
=
\alpha_0+C_k,
$$

and

$$
\beta_k
=
\beta_0+(k-C_k).
$$

Therefore, the posterior distribution becomes

$$
\Theta \mid Y^{(k)}
\sim
\mathrm{Beta}
\left(
\alpha_0+C_k,\;
\beta_0+k-C_k
\right).
$$

The corresponding posterior density is

$$
f(\theta \mid Y^{(k)})
=
\frac{
1
}{
B(\alpha_k,\beta_k)
}
\theta^{\alpha_k-1}
(1-\theta)^{\beta_k-1},
\qquad
0\leq\theta\leq1.
$$

## Posterior Mean

For a Beta distribution,

$$
\Theta
\sim
\mathrm{Beta}(\alpha_k,\beta_k),
$$

the posterior mean is

$$
E[\Theta \mid Y^{(k)}]
=
\frac{\alpha_k}{\alpha_k+\beta_k}.
$$

Substituting the closed-form update equations gives

$$
E[\Theta \mid Y^{(k)}]
=
\frac{\alpha_0+C_k}
{\alpha_0+\beta_0+k}.
$$

This expression combines the prior belief with the observed click data.

## Interpretation

The closed-form update equations make Bayesian estimation computationally efficient because only the two parameters $\alpha$ and $\beta$ need to be stored and updated after each new observation.

The posterior mean

$$
E[\Theta \mid Y^{(k)}]
=
\frac{\alpha_0+C_k}
{\alpha_0+\beta_0+k}
$$

can be interpreted as a weighted combination of

- the prior belief represented by $(\alpha_0,\beta_0)$, and
- the observed click data represented by $C_k$ and $k$.

As the number of impressions becomes very large, the influence of the prior decreases and the posterior mean gradually approaches the empirical click-through rate,

$$
\frac{C_k}{k}.
$$

This demonstrates that Bayesian estimation becomes increasingly data-driven as more observations are collected while still benefiting from prior information when only a small amount of data is available.

# Task 5 – Numerical Simulation of Bayesian CTR Tracking

To illustrate the Bayesian updating process, a numerical simulation is performed for a sequence of advertisement impressions.

Assume that the true click-through rate (CTR) of the advertisement is

$$
\theta_{\text{true}}=0.35.
$$

Initially, before any observations are made, the prior belief is chosen as a uniform Beta distribution,

$$
\Theta
\sim
\mathrm{Beta}(1,1).
$$

For each advertisement impression,

1. A click is simulated using the true click-through rate.
2. The Beta prior is updated using the observed outcome.
3. The posterior mean is calculated.
4. The posterior distribution is plotted.
5. The updated posterior becomes the prior for the next observation.

The recursive parameter updates are

$$
\alpha_k=\alpha_{k-1}+y_k,
$$

$$
\beta_k=\beta_{k-1}+(1-y_k),
$$

where

- $y_k=1$ represents a click,
- $y_k=0$ represents a non-click.

The posterior mean after the $k$-th observation is

$$
E[\Theta|Y^{(k)}]
=
\frac{\alpha_k}
{\alpha_k+\beta_k}.
$$

As more impressions are observed, the posterior distribution becomes increasingly concentrated around the true click-through rate. Consequently, the posterior mean converges towards the true CTR while the uncertainty of the estimate decreases.

In [4]:
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go

# ============================================================
# Configuration
# ============================================================

np.random.seed(42)

theta_true = 0.35          # True click-through rate
n_impressions = 100

# Initial Beta prior
alpha = 1
beta = 1

theta = np.linspace(0, 1, 500)

posterior_means = []
alpha_history = [alpha]
beta_history = [beta]

# ============================================================
# Sequential Bayesian Updating
# ============================================================

for i in range(n_impressions):

    # Simulate click
    click = np.random.binomial(1, theta_true)

    # Update Beta parameters
    alpha += click
    beta += (1 - click)

    alpha_history.append(alpha)
    beta_history.append(beta)

    # Posterior mean
    posterior_means.append(alpha / (alpha + beta))

# ============================================================
# Plot Posterior Mean Convergence
# ============================================================

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=np.arange(1, n_impressions + 1),
        y=posterior_means,
        mode="lines",
        name="Posterior Mean",
        line=dict(width=3)
    )
)

fig.add_trace(
    go.Scatter(
        x=[1, n_impressions],
        y=[theta_true, theta_true],
        mode="lines",
        name="True CTR",
        line=dict(dash="dash")
    )
)

fig.update_layout(
    title="Posterior Mean Convergence",
    xaxis_title="Number of Impressions",
    yaxis_title="Estimated CTR",
    template="plotly_white",
    width=900,
    height=550
)

fig.show()

# ============================================================
# Posterior Distribution After All Observations
# ============================================================

posterior = stats.beta.pdf(theta, alpha, beta)

fig2 = go.Figure()

fig2.add_trace(
    go.Scatter(
        x=theta,
        y=posterior,
        mode="lines",
        name=f"Posterior Beta({alpha},{beta})",
        line=dict(width=3)
    )
)

fig2.add_vline(
    x=theta_true,
    line_dash="dash",
    annotation_text="True CTR"
)

fig2.update_layout(
    title="Posterior Distribution After Bayesian Updating",
    xaxis_title="CTR (θ)",
    yaxis_title="Probability Density",
    template="plotly_white",
    width=900,
    height=550
)

fig2.show()

# ============================================================
# Final Results
# ============================================================

print(f"True CTR           : {theta_true:.3f}")
print(f"Posterior Alpha    : {alpha}")
print(f"Posterior Beta     : {beta}")
print(f"Posterior Mean     : {alpha/(alpha+beta):.4f}")

True CTR           : 0.350
Posterior Alpha    : 33
Posterior Beta     : 69
Posterior Mean     : 0.3235


# Task 6 – Interpretation and Convergence Analysis

The numerical simulation demonstrates how Bayesian inference continuously updates the estimate of the advertisement's click-through rate (CTR) as new observations become available.

Initially, the prior distribution is chosen as

$$
\Theta \sim \mathrm{Beta}(1,1),
$$

which represents complete prior uncertainty because every value of the CTR between 0 and 1 is considered equally likely.

After each advertisement impression, the posterior parameters are updated using

$$
\alpha_k=\alpha_{k-1}+y_k,
$$

and

$$
\beta_k=\beta_{k-1}+(1-y_k),
$$

where

- $y_k=1$ represents a click,
- $y_k=0$ represents a non-click.

The posterior mean after the $k$-th observation is

$$
E[\Theta|Y^{(k)}]
=
\frac{\alpha_k}
{\alpha_k+\beta_k}.
$$

As more advertisement impressions are observed, the values of $\alpha_k$ and $\beta_k$ increase, causing the posterior distribution to become progressively narrower. This reduction in spread indicates that the uncertainty associated with the estimated CTR decreases over time.

The convergence plot shows that the posterior mean gradually approaches the true click-through rate,

$$
\theta_{\text{true}}=0.35.
$$

During the early stages of the simulation, the estimate fluctuates because only a small number of observations are available. As additional click and non-click data are collected, these fluctuations become smaller and the estimate stabilises near the true CTR.

The final posterior distribution is sharply concentrated around the true click-through rate, demonstrating that the Bayesian updating procedure successfully incorporates new evidence while preserving information from previous observations.

## Interpretation

The Bayesian Beta-Binomial model provides an efficient method for tracking click-through rates in real time.

The main advantages of this approach are:

- The posterior distribution remains within the Beta family after every update, making the calculations computationally efficient.
- Only the two parameters $\alpha$ and $\beta$ need to be stored and updated as new observations arrive.
- The posterior mean provides a smooth estimate of the click-through rate by combining prior knowledge with observed data.
- As the number of observations increases, the influence of the prior distribution gradually decreases and the estimate becomes increasingly data-driven.
- The uncertainty of the estimate decreases continuously, producing a more reliable estimate of the true click-through rate.

Therefore, the Beta-Binomial Bayesian updating framework is well suited for online applications such as digital advertising, recommendation systems, and A/B testing, where click data are received sequentially and parameter estimates must be updated in real time.